# Estimate behavioral-to-UPPT002 timing offsets

This notebook compares button press times from beh_data with level changes in the matching participant's UPPT002 channel. The signed offset is UPPT002 detection time minus behavioral time: positive means the MEG channel changed later; negative means it changed earlier.

UPPT002 is an event channel. For each trial, the detector estimates baseline and noise robustly from the pre-cue interval, requires a sustained 20 ms level change away from baseline, rejects brief pulse-edge ringing, and ignores returns to baseline. Dynamic programming pairs expected and detected events in order, at most once each, and only within a configurable tolerance.

The repository does not document missing-value sentinels for A/timing; this code ignores non-finite entries only. It does not guess whether zero means a missing press.


In [ ]:
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

sub_ID = input('subject ID:')             # subject ID for participants filepath
sens_delay = 2.8                          # offset in seconds between the task's cue-time reference and the MEG epoch start
trial_time = 14.8                         # trial duration in seconds
sfreq = 1000                              # sampling frequency in Hz
n_trials = 240                            # number of trials per participant  
n_press_slots = 5                         # button presses per trial
match_tolerance = 0.25                    # time, in seconds, for largest difference between press and UPPT002 events to be considered the same event

# Adaptive robust edge detector settings.
NOISE_SIGMA_MULTIPLIER = 8.0
RELATIVE_STEP_FRACTION = 0.02
PERSISTENCE_S = 0.020
REFRACTORY_S = 0.020

def filename_loader(id):
    '''
    id = subject ID in 2 digit format, as a string
    '''

    # define file directories for behavioural data in beh_data folder
    subject_folder = "//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
    button_press_folder = '/beh_data/s'+str(id)
    BP1 = subject_folder + button_press_folder + "/behDataMEG.mat"
    BP2 = subject_folder + button_press_folder + "/targetMEG.mat"

    # define file directories for .fif MNE files in subject folder
    # folder naming convention differs for IDs that begin with 0, so handled seperately.
    if sub_ID.startswith('0'):
        spliced_id = id[1:]
        EMG_file ='/s0'+str(spliced_id)+'/Subject'+str(spliced_id)+'_spm_mne-epo'
    else:
        EMG_file ='/s'+str(id)+'/Subject'+str(id)+'_spm_mne-epo'
    CD_file = subject_folder + EMG_file + '.fif'

    # open behavioural data file 
    with h5py.File(BP1, 'r') as f:
        timings = f['A/timing'][:]     # read timing dataset into numpy array
        presses = f['A/press'][:]      # read press identity/order codes

    # open target/cue file and read cue duration data into numpy array
    with h5py.File(BP2, 'r') as f:
        cuedur = f['T/cueDur'][:]
    
    # transpose all arrays to trial-by-press layout expected 
    timings = timings.T               
    PRES = presses.T    
    cuedur = cuedur.T

    # add cue duration to each press timing and converts ms to seconds
    PT = (timings + np.tile(cuedur,n_press_slots)) / 1000

    # read the epochs file and load their samples into CD 
    CD = mne.read_epochs(CD_file, preload=True)

    # update sample frequency extracted from MNE epoch data 
    sfreq = float(CD.info["sfreq"])

    # return outputs press times, press identity codes and the MNE epochs object
    return PT,PRES,CD

press_times, presses, data = filename_loader(sub_ID)


print(f'press times array: {press_times}')
print('\n')
print(f'press times shape: {press_times.shape}')
print('\n')
print(f'press sequence array:{presses}')
print('\n')
print(f'presses array shape {presses.shape}')

print(data.info)


uppt002_chan = data.get_data(picks='UPPT002')[:,0,:]
epoch_times = data.times
cue_onset_epoch_s = epoch_times[0] + sens_delay
match_records, trial_records, detected_by_trial = [], [], []


def _robust_sigma(values):
    values = np.asarray(values, dtype=float)
    center = np.median(values)
    return 1.4826 * np.median(np.abs(values - center))

def detect_uppt002_changes(signal, epoch_times, cue_onset_epoch_s):
    """Detect persistent UPPT002 level changes away from pre-cue baseline. 
    A 20 ms persistence window rejects brief ringing at pulse endings; only
    the transition into a sustained non-baseline level is counted."""
    
    signal = np.asarray(signal, dtype=float).reshape(-1)
    epoch_times = np.asarray(epoch_times, dtype=float).reshape(-1)

    dt = float(np.median(np.diff(epoch_times)))
    sfreq = 1.0 / dt

    # Use the stable portion before cue onset to calculate baseline
    baseline_mask = (
        (epoch_times >= epoch_times[0] + 0.05)
        & (epoch_times < cue_onset_epoch_s - 0.05)
    )
    baseline_values = signal[baseline_mask]

    # find baseline signal, estimating ro
    baseline = float(np.median(baseline_values))
    baseline_noise = _robust_sigma(baseline_values)
    baseline_steps = np.diff(baseline_values)
    step_noise = _robust_sigma(baseline_steps) if baseline_steps.size else 0.0


    # Sets a minimum step size to the largest of: eight times robust sample-to-sample noise, 
    # 2% of the 99.9th-percentile signal excursion, and a tiny numerical floor. 
    # This adapts the threshold to each epoch while reducing sensitivity to isolated outliers.
    excursion = float(np.percentile(np.abs(signal - baseline), 99.9))
    numerical_floor = np.finfo(float).eps * max(1.0, float(np.max(np.abs(signal)))) * 10
    step_threshold = max(
        NOISE_SIGMA_MULTIPLIER * step_noise,
        RELATIVE_STEP_FRACTION * excursion,
        numerical_floor,
    )
    active_threshold = max(3.0 * baseline_noise, 0.25 * step_threshold, numerical_floor)


    # checks edges using 20ms median windows before and after edge.
    # transitions to baseline are not counted as new presses 
    persistence_n = max(3, int(np.ceil(PERSISTENCE_S * sfreq)))
    refractory_n = max(1, int(np.ceil(REFRACTORY_S * sfreq)))
    edge_sizes = np.abs(np.diff(signal, prepend=signal[0]))
    edge_indices = np.flatnonzero(edge_sizes >= step_threshold)

    event_samples = []
    last_event_sample = -refractory_n
    for sample in edge_indices:
        if sample < persistence_n or sample + persistence_n > signal.size:
            continue
        if sample - last_event_sample < refractory_n:
            continue

        before = float(np.median(signal[sample - persistence_n:sample]))
        after = float(np.median(signal[sample:sample + persistence_n]))
        if abs(after - before) < step_threshold:
            continue
        # A reset is a change whose post-edge level is back at baseline.
        if abs(after - baseline) <= active_threshold:
            continue

        event_samples.append(int(sample))
        last_event_sample = int(sample)

    event_samples = np.asarray(event_samples, dtype=int)
    event_times = epoch_times[event_samples] - cue_onset_epoch_s

    diagnostics = {
        "baseline": baseline,
        "baseline_noise_sigma": baseline_noise,
        "step_threshold": step_threshold,
        "active_threshold": active_threshold,
        "n_detected": len(event_samples),
    }
    return event_times, event_samples, diagnostics

def match_events_in_order(behavior_times, detected_times, tolerance_s):
    """Maximum-cardinality, minimum-offset, order-preserving one-to-one match."""
    behavior_times = np.asarray(behavior_times, dtype=float)
    detected_times = np.asarray(detected_times, dtype=float)
    behavior_order = np.flatnonzero(np.isfinite(behavior_times))
    behavior_order = behavior_order[
        np.argsort(behavior_times[behavior_order], kind="stable")
    ]
    detection_order = np.argsort(detected_times, kind="stable")
    behavior_sorted = behavior_times[behavior_order]
    detection_sorted = detected_times[detection_order]

    n, m = behavior_sorted.size, detection_sorted.size
    n_matches = np.zeros((n + 1, m + 1), dtype=int)
    total_cost = np.zeros((n + 1, m + 1), dtype=float)
    action = np.zeros((n + 1, m + 1), dtype=np.int8)
    action[1:, 0] = 1
    action[0, 1:] = 2

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            options = [
                (n_matches[i - 1, j], total_cost[i - 1, j], 1),
                (n_matches[i, j - 1], total_cost[i, j - 1], 2),
            ]
            difference = abs(behavior_sorted[i - 1] - detection_sorted[j - 1])
            if difference <= tolerance_s:
                options.append(
                    (n_matches[i - 1, j - 1] + 1,
                     total_cost[i - 1, j - 1] + difference, 3)
                )
            best = max(options, key=lambda item: (item[0], -item[1], item[2] == 3))
            n_matches[i, j], total_cost[i, j], action[i, j] = best

    pairs = []
    i, j = n, m
    while i > 0 or j > 0:
        step = action[i, j]
        if step == 3:
            pairs.append((int(behavior_order[i - 1]), int(detection_order[j - 1])))
            i -= 1
            j -= 1
        elif step == 1:
            i -= 1
        elif step == 2:
            j -= 1
        else:
            break
    return list(reversed(pairs))




def analyze_subject():
    for trial_idx, signal in enumerate(uppt002_chan):
        detected_times, _, diag = detect_uppt002_changes(
            signal, epoch_times, cue_onset_epoch_s
        )
        detected_by_trial.append(detected_times)
        expected = press_times[trial_idx]
        pairs = match_events_in_order(expected, detected_times, match_tolerance)

        for behavior_slot, detected_idx in pairs:
            behavior_time = float(expected[behavior_slot])
            uppt_time = float(detected_times[detected_idx])
            match_records.append(
                {
                    "participant": sub_ID,
                    "trial": trial_idx + 1,
                    "press_slot": behavior_slot + 1,
                    "behavior_time_s": behavior_time,
                    "uppt002_time_s": uppt_time,
                    # Sign convention: positive means the UPPT002 transition came later.
                    "offset_s": uppt_time - behavior_time,
                }
            )

        n_expected = int(np.isfinite(expected).sum())
        n_matched = len(pairs)
        trial_records.append(
            {
                "participant": sub_ID,
                "trial": trial_idx + 1,
                "n_behavior_expected": n_expected,
                "n_uppt_detected": len(detected_times),
                "n_matched": n_matched,
                "n_behavior_unmatched": n_expected - n_matched,
                "n_uppt_unmatched": len(detected_times) - n_matched,
                "baseline_noise_sigma": diag["baseline_noise_sigma"],
                "step_threshold": diag["step_threshold"],
            }
        )

    return {
        "participant": sub_ID,
        "epoch_times": epoch_times.copy(),
        "cue_onset_epoch_s": cue_onset_epoch_s,
        "behavior_times": press_times,
        "detected_by_trial": detected_by_trial,
        "matches": match_records,
        "trial_qc": trial_records,
    }


def summarize_participants(results):
"""Summarize offsets, giving each trial equal weight in the primary mean."""
match_columns = [
    "participant", "trial", "press_slot", "behavior_time_s", "uppt002_time_s", "offset_s"
]
qc_columns = [
    "participant", "trial", "n_behavior_expected", "n_uppt_detected", "n_matched",
    "n_behavior_unmatched", "n_uppt_unmatched", "baseline_noise_sigma", "step_threshold"
]
matches = pd.DataFrame(
    [row for result in results for row in result["matches"]], columns=match_columns
)
trial_qc = pd.DataFrame(
    [row for result in results for row in result["trial_qc"]], columns=qc_columns
)
summaries = []

for result in results:
    participant = result["participant"]
    participant_trials = trial_qc[trial_qc["participant"] == participant]
    participant_matches = matches[matches["participant"] == participant]
    if participant_matches.empty:
        trial_means = np.array([], dtype=float)
        event_offsets = np.array([], dtype=float)
    else:
        trial_means = (
            participant_matches.groupby("trial", sort=True)["offset_s"].mean().to_numpy()
        )
        event_offsets = participant_matches["offset_s"].to_numpy()

    n_trials = trial_means.size
    trial_mean = float(np.mean(trial_means)) if n_trials else np.nan
    trial_sd = float(np.std(trial_means, ddof=1)) if n_trials > 1 else np.nan
    trial_sem = trial_sd / np.sqrt(n_trials) if n_trials > 1 else np.nan
    expected_count = int(participant_trials["n_behavior_expected"].sum())
    matched_count = int(participant_trials["n_matched"].sum())
    summaries.append(
        {
            "participant": participant,
            "n_trials": int(len(participant_trials)),
            "n_trials_with_match": int(n_trials),
            "n_expected_events": expected_count,
            "n_matched_events": matched_count,
            "match_rate": matched_count / expected_count if expected_count else np.nan,
            # Primary result: mean of within-trial means, so each trial has equal weight.
            "mean_trial_offset_s": trial_mean,
            "median_trial_offset_s": float(np.median(trial_means)) if n_trials else np.nan,
            "sd_trial_offset_s": trial_sd,
            "approx_95ci_low_s": trial_mean - 1.96 * trial_sem if n_trials > 1 else np.nan,
            "approx_95ci_high_s": trial_mean + 1.96 * trial_sem if n_trials > 1 else np.nan,
            # Event-weighted values are useful when trials have different match counts.
            "mean_event_offset_s": float(np.mean(event_offsets)) if event_offsets.size else np.nan,
            "median_event_offset_s": float(np.median(event_offsets)) if event_offsets.size else np.nan,
            "mad_event_offset_s": (
                float(np.median(np.abs(event_offsets - np.median(event_offsets))))
                if event_offsets.size else np.nan
            ),
            "n_unmatched_behavior": int(participant_trials["n_behavior_unmatched"].sum()),
            "n_unmatched_uppt": int(participant_trials["n_uppt_unmatched"].sum()),
        }
    )

return pd.DataFrame(summaries), matches, trial_qc
    

In [ ]:
def _robust_sigma(values):
    values = np.asarray(values, dtype=float)
    center = np.median(values)
    return 1.4826 * np.median(np.abs(values - center))


def detect_uppt002_changes(signal, epoch_times, cue_onset_epoch_s):
    """Detect persistent UPPT002 level changes away from pre-cue baseline. 
    A 20 ms persistence window rejects brief ringing at pulse endings; only
    the transition into a sustained non-baseline level is counted."""
    
    signal = np.asarray(signal, dtype=float).reshape(-1)
    epoch_times = np.asarray(epoch_times, dtype=float).reshape(-1)
    if signal.size != epoch_times.size:
        raise ValueError("UPPT002 samples and epoch times must have the same length")
    if signal.size < 10 or not np.isfinite(signal).all() or not np.isfinite(epoch_times).all():
        raise ValueError("UPPT002 and epoch times must be finite and contain enough samples")

    dt = float(np.median(np.diff(epoch_times)))
    if dt <= 0:
        raise ValueError("Epoch times must be strictly increasing")
    sfreq = 1.0 / dt

    # Use the stable portion before cue onset; leave a 50 ms guard before the cue.
    baseline_mask = (
        (epoch_times >= epoch_times[0] + 0.05)
        & (epoch_times < cue_onset_epoch_s - 0.05)
    )
    baseline_values = signal[baseline_mask]
    if baseline_values.size < max(10, int(round(0.05 * sfreq))):
        raise ValueError("Too few pre-cue samples; check cue-onset and epoch timing")

    baseline = float(np.median(baseline_values))
    baseline_noise = _robust_sigma(baseline_values)
    baseline_steps = np.diff(baseline_values)
    step_noise = _robust_sigma(baseline_steps) if baseline_steps.size else 0.0
    excursion = float(np.percentile(np.abs(signal - baseline), 99.9))
    numerical_floor = np.finfo(float).eps * max(1.0, float(np.max(np.abs(signal)))) * 10
    step_threshold = max(
        NOISE_SIGMA_MULTIPLIER * step_noise,
        RELATIVE_STEP_FRACTION * excursion,
        numerical_floor,
    )
    active_threshold = max(3.0 * baseline_noise, 0.25 * step_threshold, numerical_floor)

    persistence_n = max(3, int(np.ceil(PERSISTENCE_S * sfreq)))
    refractory_n = max(1, int(np.ceil(REFRACTORY_S * sfreq)))
    edge_sizes = np.abs(np.diff(signal, prepend=signal[0]))
    edge_indices = np.flatnonzero(edge_sizes >= step_threshold)

    event_samples = []
    last_event_sample = -refractory_n
    for sample in edge_indices:
        if sample < persistence_n or sample + persistence_n > signal.size:
            continue
        if sample - last_event_sample < refractory_n:
            continue

        before = float(np.median(signal[sample - persistence_n:sample]))
        after = float(np.median(signal[sample:sample + persistence_n]))
        if abs(after - before) < step_threshold:
            continue
        # A reset is a change whose post-edge level is back at baseline.
        if abs(after - baseline) <= active_threshold:
            continue

        event_samples.append(int(sample))
        last_event_sample = int(sample)

    event_samples = np.asarray(event_samples, dtype=int)
    event_times = epoch_times[event_samples] - cue_onset_epoch_s
    diagnostics = {
        "baseline": baseline,
        "baseline_noise_sigma": baseline_noise,
        "step_threshold": step_threshold,
        "active_threshold": active_threshold,
        "n_detected": len(event_samples),
    }
    return event_times, event_samples, diagnostics


def match_events_in_order(behavior_times, detected_times, tolerance_s):
    """Maximum-cardinality, minimum-offset, order-preserving one-to-one match."""
    behavior_times = np.asarray(behavior_times, dtype=float)
    detected_times = np.asarray(detected_times, dtype=float)
    behavior_order = np.flatnonzero(np.isfinite(behavior_times))
    behavior_order = behavior_order[
        np.argsort(behavior_times[behavior_order], kind="stable")
    ]
    detection_order = np.argsort(detected_times, kind="stable")
    behavior_sorted = behavior_times[behavior_order]
    detection_sorted = detected_times[detection_order]

    n, m = behavior_sorted.size, detection_sorted.size
    n_matches = np.zeros((n + 1, m + 1), dtype=int)
    total_cost = np.zeros((n + 1, m + 1), dtype=float)
    action = np.zeros((n + 1, m + 1), dtype=np.int8)
    action[1:, 0] = 1
    action[0, 1:] = 2

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            options = [
                (n_matches[i - 1, j], total_cost[i - 1, j], 1),
                (n_matches[i, j - 1], total_cost[i, j - 1], 2),
            ]
            difference = abs(behavior_sorted[i - 1] - detection_sorted[j - 1])
            if difference <= tolerance_s:
                options.append(
                    (n_matches[i - 1, j - 1] + 1,
                     total_cost[i - 1, j - 1] + difference, 3)
                )
            best = max(options, key=lambda item: (item[0], -item[1], item[2] == 3))
            n_matches[i, j], total_cost[i, j], action[i, j] = best

    pairs = []
    i, j = n, m
    while i > 0 or j > 0:
        step = action[i, j]
        if step == 3:
            pairs.append((int(behavior_order[i - 1]), int(detection_order[j - 1])))
            i -= 1
            j -= 1
        elif step == 1:
            i -= 1
        elif step == 2:
            j -= 1
        else:
            break
    return list(reversed(pairs))

In [4]:
def _find_epochs_file(data_root, subject_id):
    """Find the main FIF file; MNE follows its -1/-2 split files automatically."""
    subject_number = int(subject_id[1:])
    search_dirs = [data_root / subject_id, data_root]
    candidates = []
    for folder in search_dirs:
        if not folder.is_dir():
            continue
        for path in folder.glob("Subject*_spm_mne-epo.fif"):
            number_text = path.name[len("Subject"):].split("_", 1)[0]
            if number_text.isdigit() and int(number_text) == subject_number:
                candidates.append(path)
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one main FIF file for {subject_id} under {data_root}, "
            f"found {len(candidates)}: {candidates}"
        )
    return candidates[0]


def analyze_subject(subject_dir):
    """Analyze one beh_data/s* folder and return event and trial QC records."""
    subject_id = subject_dir.name
    behavior_times = load_behavioral_press_times(subject_dir)

    epochs_path = _find_epochs_file(DATA_ROOT, subject_id)
    epochs = mne.read_epochs(epochs_path, preload=False, verbose="ERROR")
    if "UPPT002" not in epochs.ch_names:
        raise ValueError(f"{epochs_path} has no UPPT002 channel")
    if len(epochs) != behavior_times.shape[0]:
        raise ValueError(
            f"{subject_id}: {len(epochs)} MEG epochs but "
            f"{behavior_times.shape[0]} behavioral trials; trial matching is unsafe"
        )

    epoch_times = epochs.times
    cue_onset_epoch_s = epoch_times[0] + CUE_ONSET_AFTER_EPOCH_START_S
    channel = epochs.get_data(picks=["UPPT002"])[:, 0, :]
    match_records, trial_records, detected_by_trial = [], [], []

    for trial_idx, signal in enumerate(channel):
        detected_times, _, diag = detect_uppt002_changes(
            signal, epoch_times, cue_onset_epoch_s
        )
        detected_by_trial.append(detected_times)
        expected = behavior_times[trial_idx]
        pairs = match_events_in_order(expected, detected_times, MATCH_TOLERANCE_S)

        for behavior_slot, detected_idx in pairs:
            behavior_time = float(expected[behavior_slot])
            uppt_time = float(detected_times[detected_idx])
            match_records.append(
                {
                    "participant": subject_id,
                    "trial": trial_idx + 1,
                    "press_slot": behavior_slot + 1,
                    "behavior_time_s": behavior_time,
                    "uppt002_time_s": uppt_time,
                    # Sign convention: positive means the UPPT002 transition came later.
                    "offset_s": uppt_time - behavior_time,
                }
            )

        n_expected = int(np.isfinite(expected).sum())
        n_matched = len(pairs)
        trial_records.append(
            {
                "participant": subject_id,
                "trial": trial_idx + 1,
                "n_behavior_expected": n_expected,
                "n_uppt_detected": len(detected_times),
                "n_matched": n_matched,
                "n_behavior_unmatched": n_expected - n_matched,
                "n_uppt_unmatched": len(detected_times) - n_matched,
                "baseline_noise_sigma": diag["baseline_noise_sigma"],
                "step_threshold": diag["step_threshold"],
            }
        )

    return {
        "participant": subject_id,
        "epochs_path": epochs_path,
        "epoch_times": epoch_times.copy(),
        "cue_onset_epoch_s": cue_onset_epoch_s,
        "behavior_times": behavior_times,
        "detected_by_trial": detected_by_trial,
        "matches": match_records,
        "trial_qc": trial_records,
    }


def summarize_participants(results):
    """Summarize offsets, giving each trial equal weight in the primary mean."""
    match_columns = [
        "participant", "trial", "press_slot", "behavior_time_s", "uppt002_time_s", "offset_s"
    ]
    qc_columns = [
        "participant", "trial", "n_behavior_expected", "n_uppt_detected", "n_matched",
        "n_behavior_unmatched", "n_uppt_unmatched", "baseline_noise_sigma", "step_threshold"
    ]
    matches = pd.DataFrame(
        [row for result in results for row in result["matches"]], columns=match_columns
    )
    trial_qc = pd.DataFrame(
        [row for result in results for row in result["trial_qc"]], columns=qc_columns
    )
    summaries = []

    for result in results:
        participant = result["participant"]
        participant_trials = trial_qc[trial_qc["participant"] == participant]
        participant_matches = matches[matches["participant"] == participant]
        if participant_matches.empty:
            trial_means = np.array([], dtype=float)
            event_offsets = np.array([], dtype=float)
        else:
            trial_means = (
                participant_matches.groupby("trial", sort=True)["offset_s"].mean().to_numpy()
            )
            event_offsets = participant_matches["offset_s"].to_numpy()

        n_trials = trial_means.size
        trial_mean = float(np.mean(trial_means)) if n_trials else np.nan
        trial_sd = float(np.std(trial_means, ddof=1)) if n_trials > 1 else np.nan
        trial_sem = trial_sd / np.sqrt(n_trials) if n_trials > 1 else np.nan
        expected_count = int(participant_trials["n_behavior_expected"].sum())
        matched_count = int(participant_trials["n_matched"].sum())
        summaries.append(
            {
                "participant": participant,
                "n_trials": int(len(participant_trials)),
                "n_trials_with_match": int(n_trials),
                "n_expected_events": expected_count,
                "n_matched_events": matched_count,
                "match_rate": matched_count / expected_count if expected_count else np.nan,
                # Primary result: mean of within-trial means, so each trial has equal weight.
                "mean_trial_offset_s": trial_mean,
                "median_trial_offset_s": float(np.median(trial_means)) if n_trials else np.nan,
                "sd_trial_offset_s": trial_sd,
                "approx_95ci_low_s": trial_mean - 1.96 * trial_sem if n_trials > 1 else np.nan,
                "approx_95ci_high_s": trial_mean + 1.96 * trial_sem if n_trials > 1 else np.nan,
                # Event-weighted values are useful when trials have different match counts.
                "mean_event_offset_s": float(np.mean(event_offsets)) if event_offsets.size else np.nan,
                "median_event_offset_s": float(np.median(event_offsets)) if event_offsets.size else np.nan,
                "mad_event_offset_s": (
                    float(np.median(np.abs(event_offsets - np.median(event_offsets))))
                    if event_offsets.size else np.nan
                ),
                "n_unmatched_behavior": int(participant_trials["n_behavior_unmatched"].sum()),
                "n_unmatched_uppt": int(participant_trials["n_uppt_unmatched"].sum()),
            }
        )

    return pd.DataFrame(summaries), matches, trial_qc

In [5]:
# Analyze every available beh_data/s* folder. The matching MEG epoch file is
# expected in the same participant folder one level above beh_data.
subject_dirs = []
for behavior_root in (BEHAVIOR_ROOT, DATA_ROOT):
    subject_dirs = sorted(
        path for path in behavior_root.glob("s*")
        if path.is_dir()
        and path.name[1:].isdigit()
        and (path / "behDataMEG.mat").is_file()
        and (path / "targetMEG.mat").is_file()
    )
    if subject_dirs:
        break
if not subject_dirs:
    raise FileNotFoundError(
        f"No s* folder with behDataMEG.mat and targetMEG.mat under "
        f"{BEHAVIOR_ROOT} or {DATA_ROOT}"
    )

results = [analyze_subject(subject_dir) for subject_dir in subject_dirs]
summary_table, event_table, trial_qc_table = summarize_participants(results)

display(summary_table.round(4))
display(
    trial_qc_table.groupby("participant")[
        ["n_behavior_unmatched", "n_uppt_unmatched"]
    ].sum()
)

FileNotFoundError: No s* folder with behDataMEG.mat and targetMEG.mat under \\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data\beh_data or \\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data

In [ ]:
# Cohort QC: trial-weighted means with approximate 95% CIs and all matched
# event offsets. A negative offset means UPPT002 changed before the PC log time.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_summary = summary_table.dropna(subset=["mean_trial_offset_s"]).reset_index(drop=True)
x = np.arange(len(plot_summary))
lower = plot_summary["mean_trial_offset_s"] - plot_summary["approx_95ci_low_s"]
upper = plot_summary["approx_95ci_high_s"] - plot_summary["mean_trial_offset_s"]
axes[0].errorbar(
    x, plot_summary["mean_trial_offset_s"], yerr=np.vstack([lower, upper]),
    fmt="o", capsize=3,
)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_xticks(x, plot_summary["participant"], rotation=45)
axes[0].set_ylabel("UPPT002 − behavioral time (s)")
axes[0].set_title("Participant mean of within-trial offsets")
axes[0].grid(alpha=0.25)

if not event_table.empty:
    axes[1].hist(event_table["offset_s"], bins=40, color="slateblue", alpha=0.8)
    axes[1].axvline(0, color="black", linewidth=1)
    axes[1].axvline(event_table["offset_s"].mean(), color="darkorange", linestyle="--")
axes[1].set_xlabel("UPPT002 − behavioral time (s)")
axes[1].set_ylabel("Matched presses")
axes[1].set_title("All matched event offsets")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()


def plot_subject_waveforms(result, trial_numbers=None):
    """Overlay behavioral timestamps and detected UPPT002 changes for selected trials."""
    epochs = mne.read_epochs(result["epochs_path"], preload=False, verbose="ERROR")
    channel = epochs.get_data(picks=["UPPT002"])[:, 0, :]
    relative_times = epochs.times - result["cue_onset_epoch_s"]

    if trial_numbers is None:
        matched_trials = sorted({row["trial"] for row in result["matches"]})
        if not matched_trials:
            print(f"{result['participant']}: no matched trials to plot")
            return
        trial_means = pd.DataFrame(result["matches"]).groupby("trial")["offset_s"].mean()
        median_trial = int((trial_means - trial_means.median()).abs().idxmin())
        largest_trial = int(trial_means.abs().idxmax())
        trial_qc = pd.DataFrame(result["trial_qc"])
        unmatched_trials = trial_qc.loc[
            trial_qc["n_uppt_unmatched"] > 0, "trial"
        ].astype(int).tolist()
        trial_numbers = list(dict.fromkeys(
            [matched_trials[0], median_trial, largest_trial, *unmatched_trials[:2]]
        ))

    trial_numbers = [number for number in trial_numbers if 1 <= number <= len(channel)]
    fig, axes = plt.subplots(len(trial_numbers), 1, figsize=(13, 3.2 * len(trial_numbers)))
    axes = np.atleast_1d(axes)
    for ax, trial_number in zip(axes, trial_numbers):
        trial_idx = trial_number - 1
        ax.plot(relative_times, channel[trial_idx], color="black", linewidth=0.8)
        for slot, behavior_time in enumerate(result["behavior_times"][trial_idx]):
            if np.isfinite(behavior_time):
                ax.axvline(
                    behavior_time, color="tab:red", linestyle="--", alpha=0.7,
                    label="Behavioral time" if slot == 0 else None,
                )
        detected = result["detected_by_trial"][trial_idx]
        matched_times = np.asarray([
            row["uppt002_time_s"] for row in result["matches"]
            if row["trial"] == trial_number
        ])
        matched_mask = np.asarray([
            np.any(np.isclose(matched_times, event_time, atol=1e-9))
            for event_time in detected
        ], dtype=bool)
        if detected.size:
            values = np.interp(detected, relative_times, channel[trial_idx])
            ax.scatter(
                detected[matched_mask], values[matched_mask], marker="v",
                color="tab:green", zorder=3, label="Matched detection",
            )
            ax.scatter(
                detected[~matched_mask], values[~matched_mask], marker="x",
                color="tab:orange", zorder=3, label="Unmatched detection",
            )
        n_unmatched = int((~matched_mask).sum())
        ax.set_title(f"{result['participant']} trial {trial_number} ({n_unmatched} unmatched detections)")
        ax.set_xlabel("Time relative to cue onset (s)")
        ax.set_ylabel("UPPT002 (native units)")
        ax.grid(alpha=0.25)
        ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


# Set this to a folder name such as s09 to choose a participant; None selects the first.
QC_SUBJECT = None
qc_result = next(
    result for result in results
    if QC_SUBJECT is None or result["participant"] == QC_SUBJECT
)
plot_subject_waveforms(qc_result)